this is currently a failure; might come back to it
- main issue is filtering that i want to address first

In [4]:
# Simple outage recovery baseline:
# Predict t90 (hours from event_start to 90% recovery)
# using storm-level weather summaries + county context.
# Evaluation: leave-one-storm-out cross-validation.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [6]:
path = r"C:\Users\teaching\Downloads\outage-recovery-forecasting\data_transients"

model_df = pd.read_parquet(f"{path}\\florida_model_df.parquet")
event_master = pd.read_parquet(f"{path}\\florida_event_master.parquet")

# This CSV is expected to contain county-level context such as county_pop.
fl_pous = pd.read_csv(r"C:\Users\teaching\Downloads\outage-recovery-forecasting\data_transients\FL_POUS.csv")

print("model_df shape:", model_df.shape)
print("event_master shape:", event_master.shape)
print("fl_pous shape:", fl_pous.shape)

print("\nmodel_df columns:")
print(model_df.columns.tolist())

print("\nfl_pous columns:")
print(fl_pous.columns.tolist())

model_df shape: (17510, 22)
event_master shape: (99, 5)
fl_pous shape: (99, 13)

model_df columns:
['event_id', 'storm', 'geoid', 'county', 'datetime', 'event_start', 'duration_hours', 'gust_mps', 'wind_speed_mps', 'precip_mm', 'pressure_hpa', 'temp_c', 'CountyFIPS', 'outageFraction', 'customersTracked', 'persistence_1h', 'error', 'abs_error', 'sq_error', 'persistence_24h', 'persistence_48h', 'persistence_72h']

fl_pous columns:
['event_start', 'CountyFIPS', 'county_pop', 'pre_outage_tracked_customers', 'days_since_data_start', 'duration_hours', 'n_periods', 'integral', 'pop_hours_supply_lost', 'storm', 'duration_days', 'year', 'month']


In [20]:
df = model_df.copy()

# Standardise types
df["datetime"] = pd.to_datetime(df["datetime"])
df["event_start"] = pd.to_datetime(df["event_start"])

# IMPORTANT: CountyFIPS as zero-padded string
df["CountyFIPS"] = df["CountyFIPS"].astype(str).str.zfill(5)

# Quality filter
df = df.loc[df["customersTracked"] >= 0.30].copy()

# Sanity check
required_cols = [
    "event_id", "storm", "CountyFIPS", "county", "datetime", "event_start",
    "duration_hours", "gust_mps", "wind_speed_mps", "precip_mm",
    "pressure_hpa", "outageFraction", "customersTracked"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

rows = []

for (event_id, county_fips), g in df.groupby(["event_id", "CountyFIPS"], sort=False):
    g = g.sort_values("datetime").copy()

    # Check for hourly gaps (keep this strict for now)
    diffs = g["datetime"].diff().dropna()
    if not (diffs == pd.Timedelta(hours=1)).all():
        print(f"\n⚠️ Gap detected: event_id={event_id}, CountyFIPS={county_fips}")
        print(g["datetime"].head())
        continue  # skip for now instead of crashing

    event_start = g["event_start"].iloc[0]
    duration_hours = int(g["duration_hours"].iloc[0])

    # Target: first time after event_start when outageFraction <= 0.1
    post = g.loc[g["datetime"] >= event_start].copy()
    hit = post.loc[post["outageFraction"] <= 0.10]

    if hit.empty:
        t90 = duration_hours
        t90_censored = 1
    else:
        t90 = (hit["datetime"].iloc[0] - event_start).total_seconds() / 3600.0
        t90_censored = 0

    # First 7 days or full series if shorter
    first_168 = g.iloc[:168].copy()

    rows.append({
        "event_id": event_id,
        "storm": g["storm"].iloc[0],
        "CountyFIPS": county_fips,
        "county": g["county"].iloc[0],
        "event_start": event_start,
        "duration_hours": duration_hours,
        "t90": float(t90),
        "t90_censored": int(t90_censored),
        "max_gust": float(first_168["gust_mps"].max()),
        "mean_gust_7d": float(first_168["gust_mps"].mean()),
        "total_precip_7d": float(first_168["precip_mm"].sum()),
        "pressure_min_7d": float(first_168["pressure_hpa"].min()),
        "max_customers_tracked": float(g["customersTracked"].max()),
    })

base_df = pd.DataFrame(rows)

print(base_df.shape)
print(base_df.head())


⚠️ Gap detected: event_id=12007_2017-09-11 05:00:00, CountyFIPS=12007
872   2017-09-10 17:00:00
873   2017-09-10 18:00:00
874   2017-09-10 19:00:00
875   2017-09-10 20:00:00
876   2017-09-10 21:00:00
Name: datetime, dtype: datetime64[ns]

⚠️ Gap detected: event_id=12011_2017-09-10 06:00:00, CountyFIPS=12011
1148   2017-09-09 19:00:00
1158   2017-09-10 05:00:00
1166   2017-09-10 13:00:00
1176   2017-09-10 23:00:00
1177   2017-09-11 00:00:00
Name: datetime, dtype: datetime64[ns]

⚠️ Gap detected: event_id=12013_2017-09-09 21:00:00, CountyFIPS=12013
1360   2017-09-09 20:00:00
1361   2017-09-09 21:00:00
1363   2017-09-09 23:00:00
1364   2017-09-10 00:00:00
1366   2017-09-10 02:00:00
Name: datetime, dtype: datetime64[ns]

⚠️ Gap detected: event_id=12015_2017-09-11 00:00:00, CountyFIPS=12015
2167   2017-09-10 13:00:00
2177   2017-09-10 23:00:00
2178   2017-09-11 00:00:00
2179   2017-09-11 01:00:00
2180   2017-09-11 02:00:00
Name: datetime, dtype: datetime64[ns]

⚠️ Gap detected: event_id=12

why so many 'gaps'?

In [21]:
import pandas as pd

problem_event_id = "12007_2017-09-11 05:00:00"
problem_county = "12007"

raw = model_df.copy()
raw["datetime"] = pd.to_datetime(raw["datetime"])
raw["event_start"] = pd.to_datetime(raw["event_start"])
raw["CountyFIPS"] = raw["CountyFIPS"].astype(str).str.zfill(5)

# Full raw window, before any customersTracked filter
g_raw = raw.loc[
    (raw["event_id"] == problem_event_id) &
    (raw["CountyFIPS"] == problem_county)
].sort_values("datetime").copy()

print("RAW rows:", len(g_raw))
print("RAW start:", g_raw["datetime"].min())
print("RAW end:", g_raw["datetime"].max())

full_range = pd.date_range(g_raw["datetime"].min(), g_raw["datetime"].max(), freq="h")
missing_raw = full_range.difference(pd.Index(g_raw["datetime"]))

print("\nMissing in raw window:")
print(missing_raw)

# Now apply the quality filter
g_filt = g_raw.loc[g_raw["customersTracked"] >= 0.30].copy()

print("\nFILTERED rows:", len(g_filt))
print("Filtered start:", g_filt["datetime"].min())
print("Filtered end:", g_filt["datetime"].max())

missing_filt = full_range.difference(pd.Index(g_filt["datetime"]))

print("\nMissing after filtering:")
print(missing_filt)

# Show which timestamps were removed by the filter
removed = g_raw.loc[~g_raw["datetime"].isin(g_filt["datetime"]), [
    "datetime", "customersTracked", "outageFraction", "gust_mps", "wind_speed_mps", "precip_mm", "pressure_hpa"
]].copy()

print("\nRows removed by the filter:")
print(removed.to_string(index=False))

RAW rows: 189
RAW start: 2017-09-10 17:00:00
RAW end: 2017-09-18 13:00:00

Missing in raw window:
DatetimeIndex([], dtype='datetime64[ns]', freq='h')

FILTERED rows: 181
Filtered start: 2017-09-10 17:00:00
Filtered end: 2017-09-18 13:00:00

Missing after filtering:
DatetimeIndex(['2017-09-10 22:00:00', '2017-09-13 19:00:00',
               '2017-09-13 20:00:00', '2017-09-16 16:00:00',
               '2017-09-16 17:00:00', '2017-09-16 18:00:00',
               '2017-09-16 19:00:00', '2017-09-16 20:00:00'],
              dtype='datetime64[ns]', freq=None)

Rows removed by the filter:
           datetime  customersTracked  outageFraction  gust_mps  wind_speed_mps  precip_mm  pressure_hpa
2017-09-10 22:00:00               0.0             0.0 18.453369        8.380640   2.635949   1008.150772
2017-09-13 19:00:00               0.0             0.0  6.156727        2.250506   0.027458   1013.164409
2017-09-13 20:00:00               0.0             0.0  6.287363        2.329248   0.011854   101

In [ ]:
# Adjust these keys if the CSV uses slightly different names.
possible_keys = [c for c in fl_pous.columns if c.lower() in {"countyfips", "fips", "geoid"}]
print("Possible merge keys in FL_POUS:", possible_keys)

possible_pop_cols = [c for c in fl_pous.columns if "pop" in c.lower()]
print("Possible population columns in FL_POUS:", possible_pop_cols)

# Standardise likely merge columns if needed.
# If the column names are already exactly CountyFIPS and county_pop, this will just work.
left = base_df.copy()
right = fl_pous.copy()

# Try to find the county-pop column.
county_pop_col = None
for c in right.columns:
    if c.lower() == "county_pop":
        county_pop_col = c
        break
if county_pop_col is None:
    raise ValueError("Could not find a 'county_pop' column in FL_POUS.csv.")

# Try to find the county FIPS column.
county_fips_col = None
for c in right.columns:
    if c.lower() in {"countyfips", "fips", "geoid"}:
        county_fips_col = c
        break
if county_fips_col is None:
    raise ValueError("Could not find a CountyFIPS/FIPS/Geoid column in FL_POUS.csv.")

right = right.rename(columns={county_fips_col: "CountyFIPS", county_pop_col: "county_pop"})

# Keep only the merge columns plus county_pop.
right = right[["CountyFIPS", "county_pop"]].drop_duplicates()

# Merge county population into the modelling frame.
model_df2 = left.merge(right, on="CountyFIPS", how="left")

if model_df2["county_pop"].isna().any():
    missing = model_df2.loc[model_df2["county_pop"].isna(), "CountyFIPS"].unique()
    raise ValueError(f"Missing county_pop for CountyFIPS values: {missing}")

# Final feature/target table.
feature_cols = [
    "max_gust",
    "mean_gust_7d",
    "total_precip_7d",
    "pressure_min_7d",
    "max_customers_tracked",
    "county_pop",
    "CountyFIPS",
]

target_col = "t90"
group_col = "storm"

final_df = model_df2[feature_cols + [target_col, "t90_censored", group_col]].copy()

# Basic checks.
print(final_df.shape)
print(final_df.isna().sum())
print(final_df.head())

In [ ]:
print(final_df[target_col].describe())
print("\nCensored fraction:", final_df["t90_censored"].mean())

plt.figure(figsize=(8, 4))
plt.hist(final_df["t90"], bins=20, edgecolor="black")
plt.xlabel("t90 (hours from event_start)")
plt.ylabel("Count")
plt.title("Target distribution")
plt.tight_layout()
plt.show()

num_cols = [c for c in feature_cols if c != "CountyFIPS"]
print("\nNumeric feature correlations:")
print(final_df[num_cols + [target_col]].corr(numeric_only=True)[target_col].sort_values(ascending=False))

In [ ]:
X = final_df[feature_cols].copy()
y = final_df[target_col].copy()
groups = final_df[group_col].copy()

numeric_features = [c for c in feature_cols if c != "CountyFIPS"]
categorical_features = ["CountyFIPS"]

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features),
    ]
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("regressor", LinearRegression()),
])

logo = LeaveOneGroupOut()
preds = np.full(len(final_df), np.nan, dtype=float)

fold_rows = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups=groups), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    storm_test = groups.iloc[test_idx].iloc[0]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    preds[test_idx] = y_pred

    fold_rows.append({
        "fold": fold,
        "storm_test": storm_test,
        "n_test": len(test_idx),
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": mean_squared_error(y_test, y_pred, squared=False),
    })

fold_summary = pd.DataFrame(fold_rows)
fold_summary

In [ ]:
final_df["t90_pred"] = preds
final_df["residual"] = final_df["t90_pred"] - final_df["t90"]

print("Overall MAE:", mean_absolute_error(final_df["t90"], final_df["t90_pred"]))
print("Overall RMSE:", mean_squared_error(final_df["t90"], final_df["t90_pred"], squared=False))

plt.figure(figsize=(6, 6))
plt.scatter(final_df["t90"], final_df["t90_pred"], alpha=0.8)
m = max(final_df["t90"].max(), final_df["t90_pred"].max())
plt.plot([0, m], [0, m], linestyle="--")
plt.xlabel("Actual t90")
plt.ylabel("Predicted t90")
plt.title("Leave-one-storm-out linear baseline")
plt.tight_layout()
plt.show()

fold_summary.sort_values("mae")